# Benchmark workflow

This tutorial will show you the basic workflow of `open-qbench`: how to define samplers, analyses and run benchmarks.

## Defining samplers
There are several different samplers you can use with `open-qbench`. In this tutorial we will use Qiskit samplers, but you can also use the built-in OrcaSampler (provided you have ptseries installed) and dwave samplers or Qlauncher (Algorithm, Backend) tuples for optimization benchmarks.

In [1]:
from qiskit.primitives import BackendSamplerV2, StatevectorSampler
from qiskit_ibm_runtime.fake_provider import FakeAlmadenV2

# Define 2 samplers for the fidelity benchmark
perfect_sampler = StatevectorSampler()
backend_sampler = BackendSamplerV2(backend=FakeAlmadenV2())

## Defining analyses
Each benchmark needs an analysis to run. In this tutorial we will define a normalized-fidelity analysis between the two backends.

In [2]:
from open_qbench.analysis import FidelityAnalysis
from open_qbench.metrics.fidelities import normalized_fidelity

analysis = FidelityAnalysis(normalized_fidelity)

## Defining a benchmark
With the analysis ready, we can define a benchmark. For comparisons between two backends we will use the `ApplicationBenchmark`.  
The benchmark needs to be provided with two backends, an analysis and a circuit to run on both backends.

In [3]:
from open_qbench.apps.circuits import ghz_direct
from open_qbench.benchmarks import ApplicationBenchmark
from open_qbench.core import BenchmarkInput

circuit = ghz_direct(num_qubits=5)

benchmark = ApplicationBenchmark(
    backend_sampler=backend_sampler,
    reference_state_sampler=perfect_sampler,
    benchmark_input=BenchmarkInput(program=circuit),
    analysis=analysis,
    name="GHZ_Fidelity",
)

Now we can run the benchmark and get our fidelity.

In [4]:
result = benchmark.run()

print(f"Normalized fidelity: {result.metrics['fidelity']}")

Normalized fidelity: 0.5487655214131502


## Running multiple benchmarks at once
`BenchmarkManager` can be used to run multiple benchmarks at once.

In [ ]:
from open_qbench import BenchmarkManager

manager = BenchmarkManager()

for i in range(2, 6):
    circuit = ghz_direct(num_qubits=i)

    benchmark = ApplicationBenchmark(
        backend_sampler=backend_sampler,
        reference_state_sampler=perfect_sampler,
        benchmark_input=BenchmarkInput(program=circuit),
        analysis=analysis,
        name="GHZ_Fidelity",
    )
    manager.add_benchmarks(benchmark)

manager.run_all()

print("Results:")
[(res.input.program.num_qubits, res.metrics["fidelity"]) for res in manager.results]

Results:


[(2, 0.6489382653362371),
 (3, 0.6595283373531935),
 (4, 0.5633919884990137),
 (5, 0.5675952757050089)]